# Live Demo 3 — Generative Adversarial Network: DCGAN on FashionMNIST (CPU)

**Goal (≈ 60–75 min incl. discussion):** train a small DCGAN *live on a laptop CPU* and watch it go from pure noise to
recognisable shirts, trousers, sneakers and bags in a few minutes.

| Step | What students see | Approx. time (reference laptop, see README) |
|---|---|---|
| 1 | Real data grid | < 5 s |
| 2 | Generator & discriminator, DCGAN weight init | < 1 s |
| 3 | Training: loss prints + fixed-noise sample grid after every epoch | ≈ 8–10 min (or ~1 s with `USE_CACHED=True`) |
| 4 | Loss curves (and why they do *not* tell you image quality) | < 5 s |
| 5 | Animated GIF of training progress, latent-space interpolation | < 10 s |

> **Trainer note — safety net.** `USE_CACHED = True` loads `checkpoints/dcgan_fashionmnist_G.pth` (+ history and the
> per-epoch grids) if present. Set `USE_CACHED = False` to train live.

## 0. Setup

In [ ]:
from pathlib import Path
import time, math, random
import numpy as np
import torch, torch.nn as nn
import torchvision
from torchvision import datasets, transforms
from torchvision.utils import make_grid
import matplotlib.pyplot as plt
from PIL import Image

USE_CACHED = True           # <- set False to train live (~8-10 min on a modern laptop CPU)
SEED = 0
DATA_DIR = Path("..") / "data"
CKPT_DIR = Path("checkpoints"); CKPT_DIR.mkdir(exist_ok=True)
OUT_DIR  = Path("outputs");     OUT_DIR.mkdir(exist_ok=True)
CKPT_PATH = CKPT_DIR / "dcgan_fashionmnist_G.pth"

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print(f"torch {torch.__version__} | threads {torch.get_num_threads()} | checkpoint exists: {CKPT_PATH.exists()}")

## 1. Data: FashionMNIST (60 000 grey 28×28 images, 10 clothing classes)

We ignore the labels — a GAN is **unsupervised**: it learns the data distribution p(x).
Images are scaled to **[-1, 1]** because the generator ends with `tanh`.

In [ ]:
tfm = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])   # [0,1] -> [-1,1]
train_set = datasets.FashionMNIST(str(DATA_DIR), train=True, download=True, transform=tfm)
BATCH_SIZE = 128
loader = torch.utils.data.DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, drop_last=True,
                                     num_workers=0, generator=torch.Generator().manual_seed(SEED))
print(f"{len(train_set)} images, {len(loader)} iterations/epoch at batch {BATCH_SIZE}")

def show_grid(x, title="", nrow=8, path=None, ax=None):
    grid = make_grid(x[:nrow * nrow], nrow=nrow, normalize=True, value_range=(-1, 1), pad_value=1)
    own = ax is None
    if own: plt.figure(figsize=(5, 5)); ax = plt.gca()
    ax.imshow(grid.permute(1, 2, 0).numpy()); ax.axis("off"); ax.set_title(title)
    if path is not None: plt.savefig(path, dpi=100, bbox_inches="tight")
    if own: plt.show()

real_batch, _ = next(iter(loader))
show_grid(real_batch, "Real FashionMNIST images", path=OUT_DIR / "gan_real_samples.png")

## 2. The two networks

**DCGAN guidelines (Radford et al., 2015)** — and where they appear below:
1. No pooling: **strided convolutions** (D) and **transposed convolutions** (G) learn their own up/down-sampling.
2. **BatchNorm** in both nets (not on G's output layer, not on D's input layer).
3. No fully-connected hidden layers.
4. **ReLU** in G (tanh at the output), **LeakyReLU(0.2)** in D.
5. Weights initialised from N(0, 0.02); **Adam, lr = 2e-4, β₁ = 0.5** (β₁ = 0.9 makes training oscillate).

Shapes (28×28 version):
```
G:  z[100×1×1] ─ConvT 7×7─► 256×7×7 ─ConvT 4×4 s2─► 128×14×14 ─ConvT 4×4 s2─► 1×28×28 ─ tanh
D:  x[1×28×28] ─Conv 4×4 s2─► 64×14×14 ─Conv 4×4 s2─► 128×7×7 ─Conv 7×7─► 1 logit (real/fake)
```

In [ ]:
NZ, NGF, NDF = 100, 64, 64

class Generator(nn.Module):
    def __init__(self, nz=NZ, ngf=NGF):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(nz, ngf * 4, 7, 1, 0, bias=False), nn.BatchNorm2d(ngf * 4), nn.ReLU(True),       # 7x7
            nn.ConvTranspose2d(ngf * 4, ngf * 2, 4, 2, 1, bias=False), nn.BatchNorm2d(ngf * 2), nn.ReLU(True),  # 14x14
            nn.ConvTranspose2d(ngf * 2, 1, 4, 2, 1, bias=False), nn.Tanh())                                     # 28x28
    def forward(self, z):
        return self.net(z)

class Discriminator(nn.Module):
    def __init__(self, ndf=NDF):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, ndf, 4, 2, 1, bias=False), nn.LeakyReLU(0.2, True),                                   # 14x14
            nn.Conv2d(ndf, ndf * 2, 4, 2, 1, bias=False), nn.BatchNorm2d(ndf * 2), nn.LeakyReLU(0.2, True),    # 7x7
            nn.Conv2d(ndf * 2, 1, 7, 1, 0, bias=False), nn.Flatten())                                          # 1 logit
    def forward(self, x):
        return self.net(x).squeeze(1)

def dcgan_init(m):
    if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
        nn.init.normal_(m.weight, 0.0, 0.02)
    elif isinstance(m, nn.BatchNorm2d):
        nn.init.normal_(m.weight, 1.0, 0.02); nn.init.zeros_(m.bias)

G, D = Generator(), Discriminator()
G.apply(dcgan_init); D.apply(dcgan_init)
count = lambda m: sum(p.numel() for p in m.parameters())
print(f"Generator: {count(G)/1e6:.2f} M params | Discriminator: {count(D)/1e6:.3f} M params")
with torch.no_grad():
    z = torch.randn(2, NZ, 1, 1)
    print("G(z):", tuple(G(z).shape), "| D(G(z)):", tuple(D(G(z)).shape))

## 3. The adversarial game

$$\min_G \max_D \; \mathbb{E}_{x\sim p_{data}}[\log D(x)] + \mathbb{E}_{z\sim p(z)}[\log(1 - D(G(z)))]$$

Each iteration:
1. **Update D** (maximise): BCE on real images with label 1 + BCE on `G(z).detach()` with label 0.
   `detach()` = don't push gradients into G during D's step.
2. **Update G** with the **non-saturating** loss: BCE(D(G(z)), label 1) — i.e. "make D say *real*".
   (Minimising log(1−D(G(z))) directly gives vanishing gradients early on, when D easily rejects fakes.)

We use `BCEWithLogitsLoss` = sigmoid + BCE in one numerically-stable op, so D outputs raw logits.

We also track **D(x)** (D's average probability on real images) and **D(G(z))** — at the theoretical equilibrium both → 0.5.
A **fixed noise vector** is used to render the same 64 samples after every epoch so the class can watch them evolve.

In [ ]:
EPOCHS, LR, BETAS, PRINT_EVERY = 10, 2e-4, (0.5, 0.999), 100
fixed_noise = torch.randn(64, NZ, 1, 1, generator=torch.Generator().manual_seed(123))

def sample_grid(G, z):
    G.eval()
    with torch.no_grad():
        x = G(z)
    G.train()
    return x

def train_gan(G, D):
    crit = nn.BCEWithLogitsLoss()
    optG = torch.optim.Adam(G.parameters(), lr=LR, betas=BETAS)
    optD = torch.optim.Adam(D.parameters(), lr=LR, betas=BETAS)
    hist = {"lossD": [], "lossG": [], "Dx": [], "DGz": [], "epoch_time_s": []}
    grids = [sample_grid(G, fixed_noise)]                       # epoch 0 = untrained generator
    t_start = time.perf_counter()
    for epoch in range(1, EPOCHS + 1):
        t_ep = time.perf_counter()
        for it, (real, _) in enumerate(loader, 1):
            b = real.size(0)
            ones, zeros = torch.ones(b), torch.zeros(b)
            # ---- (1) Discriminator step ----
            z = torch.randn(b, NZ, 1, 1)
            fake = G(z)
            out_real, out_fake = D(real), D(fake.detach())
            lossD = crit(out_real, ones) + crit(out_fake, zeros)
            optD.zero_grad(); lossD.backward(); optD.step()
            # ---- (2) Generator step (non-saturating loss) ----
            out = D(fake)
            lossG = crit(out, ones)
            optG.zero_grad(); lossG.backward(); optG.step()

            hist["lossD"].append(lossD.item()); hist["lossG"].append(lossG.item())
            hist["Dx"].append(torch.sigmoid(out_real).mean().item()); hist["DGz"].append(torch.sigmoid(out_fake).mean().item())
            if it % PRINT_EVERY == 0:
                print(f"[epoch {epoch}/{EPOCHS} it {it:3d}/{len(loader)}] lossD={lossD.item():.3f} lossG={lossG.item():.3f} "
                      f"D(x)={hist['Dx'][-1]:.2f} D(G(z))={hist['DGz'][-1]:.2f}", flush=True)
        dt = time.perf_counter() - t_ep
        hist["epoch_time_s"].append(dt)
        grids.append(sample_grid(G, fixed_noise))
        show_grid(grids[-1], f"Fixed-noise samples after epoch {epoch}", path=OUT_DIR / f"gan_epoch_{epoch:02d}.png")
        print(f"==> epoch {epoch} took {dt:.0f}s", flush=True)
    hist["total_time_s"] = time.perf_counter() - t_start
    return hist, grids

if USE_CACHED and CKPT_PATH.exists():
    ckpt = torch.load(CKPT_PATH, map_location="cpu", weights_only=True)
    G.load_state_dict(ckpt["G"]); D.load_state_dict(ckpt["D"])
    hist, grids = ckpt["history"], list(ckpt["grids"])
    print(f"Loaded cached GAN (originally trained {len(hist['epoch_time_s'])} epochs in {hist['total_time_s']/60:.1f} min)")
    show_grid(grids[-1], f"Fixed-noise samples after epoch {len(grids)-1} (cached)")
else:
    print(f"Training DCGAN for {EPOCHS} epochs on CPU ...")
    hist, grids = train_gan(G, D)
    torch.save({"G": G.state_dict(), "D": D.state_dict(), "history": hist, "grids": torch.stack(grids),
                "nz": NZ, "ngf": NGF, "ndf": NDF, "fixed_noise": fixed_noise}, CKPT_PATH)
    print(f"Total training time {hist['total_time_s']/60:.1f} min -> saved {CKPT_PATH}")

## 4. Loss curves — and why they are **not** a quality metric

* The two losses fight each other: when D gets better, lossG goes up and vice-versa. There is **no single number that
  goes down** as images get better — unlike classification.
* A healthy run: both losses oscillate but stay bounded; D(x) stays somewhat above D(G(z)) without either collapsing
  to 0 or 1.
* **Failure signatures:** lossD → 0 (D wins, G gets no gradient), sudden lossG explosion, or all samples looking the same
  (**mode collapse**) even though losses look "fine".
* That is why GAN papers report sample-based metrics (FID, Inception Score, precision/recall of distributions) and
  why you should **always look at the samples**.

In [ ]:
def smooth(v, k=50):
    v = np.asarray(v); return np.convolve(v, np.ones(k) / k, mode="valid")

ipe = len(hist["lossD"]) // len(hist["epoch_time_s"])
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(hist["lossD"], alpha=.2, color="C0"); axes[0].plot(smooth(hist["lossD"]), color="C0", label="loss D")
axes[0].plot(hist["lossG"], alpha=.2, color="C1"); axes[0].plot(smooth(hist["lossG"]), color="C1", label="loss G")
axes[1].plot(smooth(hist["Dx"]), label="D(x)  (real)"); axes[1].plot(smooth(hist["DGz"]), label="D(G(z))  (fake)")
axes[1].axhline(0.5, ls="--", color="gray", label="equilibrium 0.5")
for ax, t in zip(axes, ["Losses (50-iter moving avg)", "Discriminator outputs"]):
    for e in range(1, len(hist["epoch_time_s"])): ax.axvline(e * ipe, color="k", alpha=.08)
    ax.set_xlabel("iteration"); ax.legend(); ax.set_title(t)
plt.tight_layout(); plt.savefig(OUT_DIR / "gan_loss_curves.png", dpi=90); plt.show()
print("epoch times (s):", [round(t) for t in hist["epoch_time_s"]], "| total", round(hist["total_time_s"] / 60, 1), "min")

## 5. Watching it learn — the same 64 noise vectors, epoch by epoch
Saved as `outputs/gan_progress.gif` (great to put on a slide).

In [ ]:
frames = []
for e, g in enumerate(grids):
    grid = make_grid(g, nrow=8, normalize=True, value_range=(-1, 1), pad_value=1)
    im = Image.fromarray((grid.permute(1, 2, 0).numpy() * 255).astype(np.uint8)).resize((480, 480), Image.NEAREST)
    frames.append(im)
frames[0].save(OUT_DIR / "gan_progress.gif", save_all=True, append_images=frames[1:] + [frames[-1]] * 3,
               duration=600, loop=0)
print("saved", OUT_DIR / "gan_progress.gif", f"({len(frames)} frames)")

sel = sorted(set([0, 1, 2, len(grids) // 2, len(grids) - 1]))
fig, axes = plt.subplots(1, len(sel), figsize=(4 * len(sel), 4.3))
for ax, e in zip(axes, sel):
    show_grid(grids[e], f"epoch {e}", ax=ax)
plt.tight_layout(); plt.savefig(OUT_DIR / "gan_progress_strip.png", dpi=90); plt.show()

## 6. Real vs generated
Ask the class: *which side is real?* Where does the generator still fail? (Typical: fine textures/prints on shirts,
thin straps of sandals, bags with odd handles.)

In [ ]:
torch.manual_seed(7)
fake = sample_grid(G, torch.randn(64, NZ, 1, 1))
fig, axes = plt.subplots(1, 2, figsize=(11, 5.6))
show_grid(real_batch, "real", ax=axes[0]); show_grid(fake, "generated (DCGAN)", ax=axes[1])
plt.tight_layout(); plt.savefig(OUT_DIR / "gan_real_vs_fake.png", dpi=90); plt.show()

## 7. Latent-space interpolation

If G has learned a smooth mapping, walking in a straight line between two noise vectors z₁ → z₂ gives a smooth
*morph* between two garments (e.g. shirt → dress → trouser) — evidence that the model is **not just memorising** the
training images. We use spherical interpolation (slerp), which suits Gaussian latents better than a straight line
(a linear midpoint has an atypically small norm).

In [ ]:
def slerp(z1, z2, t):
    a, b = z1.flatten(), z2.flatten()
    omega = torch.acos(torch.clamp(torch.dot(a / a.norm(), b / b.norm()), -1, 1))
    so = torch.sin(omega)
    if so.abs() < 1e-6:
        return (1 - t) * z1 + t * z2
    return (torch.sin((1 - t) * omega) / so) * z1 + (torch.sin(t * omega) / so) * z2

g = torch.Generator().manual_seed(2024)
rows, STEPS = 6, 10
zs = []
for r in range(rows):
    z1, z2 = torch.randn(NZ, 1, 1, generator=g), torch.randn(NZ, 1, 1, generator=g)
    zs += [slerp(z1, z2, t) for t in torch.linspace(0, 1, STEPS)]
imgs = sample_grid(G, torch.stack(zs))
grid = make_grid(imgs, nrow=STEPS, normalize=True, value_range=(-1, 1), pad_value=1)
plt.figure(figsize=(12, 7.5)); plt.imshow(grid.permute(1, 2, 0).numpy()); plt.axis("off")
plt.title("Latent interpolation (each row: z1 -> z2)")
plt.savefig(OUT_DIR / "gan_latent_interpolation.png", dpi=90, bbox_inches="tight"); plt.show()

## 8. Wrap-up — what to emphasise

* A GAN = **two networks in a minimax game**; the generator never sees real images, only D's gradients.
* Architecture & optimiser details (DCGAN guidelines, Adam β₁=0.5, BatchNorm, LeakyReLU) matter a lot for stability.
* **Losses don't measure quality** → look at samples, use FID etc.
* **Mode collapse & instability** are the classic failure modes (see README talking points) — modern fixes: WGAN-GP,
  spectral normalisation, TTUR, StyleGAN-style architectures; and diffusion models have largely replaced GANs for
  high-quality image generation (but GANs are still ~100× faster at sampling: one forward pass!).
* **Next:** `04_gan_optimization.ipynb` — make this generator smaller & faster for deployment.

In [ ]:
print(f"Generator checkpoint: {CKPT_PATH.resolve()}")
print("Outputs:", sorted(p.name for p in OUT_DIR.glob('gan_*')))